# Train the ORB breakout classifier

Loads the labeled dataset written by `orb_datagen.py` (Object Store key
`orb_training_data`), trains a classifier to predict P(a raw breakout hits
TP before SL), and saves it (Object Store key `orb_ml_model`) for
`orb_ml.py` to load at inference time.

**Run `orb_datagen.py` as a backtest in this same QuantConnect project
first** - Object Store is project-scoped, so this notebook can only see
data written by a backtest that ran in this project.

In [ ]:
qb = QuantBook()

import pandas as pd
import numpy as np
import io

csv_text = qb.ObjectStore.Read("orb_training_data")
df = pd.read_csv(io.StringIO(csv_text))
print(df.shape)
df.head()

## Clean up

Early rows (before indicators like the 200-bar trend MA or 100-bar ATR
baseline have enough history) have blank feature values in the CSV -
drop them rather than imputing, since imputing a "no data yet" gap with a
guessed value would quietly teach the model something false.

In [ ]:
feature_cols = [
    "range_size_pips", "range_vs_atr", "impulse_ratio", "atr_regime_ratio",
    "trend_strength", "volume_ratio", "hour_of_day", "minute_of_hour",
    "day_of_week", "direction",
]

for col in feature_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Per-column missing counts BEFORE dropping - if one column is blank on
# ~every row (volume_ratio is the likely suspect, given forex has no real
# centralized volume - see orb.py's VOLUME_FILTER comment), dropna() below
# would otherwise silently wipe the whole dataset and the only symptom
# would be a confusing "0 rows" further down. Check this first.
print("missing values per feature column:")
print(df[feature_cols].isna().sum())
print()

before = len(df)
df = df.dropna(subset=feature_cols + ["label"]).reset_index(drop=True)
print(f"{before} rows -> {len(df)} usable after dropping incomplete-feature rows")
if len(df) == 0:
    print("\nAll rows dropped - check the per-column counts above. If volume_ratio "
          "is the culprit, drop it from feature_cols (both here and in orb_ml.py's "
          "feature_row) and re-run rather than trying to fix the data - it likely "
          "means Oanda forex data has no usable volume for this symbol/resolution.")
else:
    display(df["label"].value_counts(normalize=True))

## Time-ordered train/test split

The CSV is written in the order the backtest encountered trades -
chronological. **Do not use a random split** (e.g. sklearn's
`train_test_split` with `shuffle=True`) - that would let the model train on
trades that happened after some of its test examples, which is lookahead
bias by another name. Train on the earlier 70%, test on the most recent
30%, exactly like every other backtest in this project has to respect
walk-forward ordering.

In [ ]:
split_idx = int(len(df) * 0.7)
train_df = df.iloc[:split_idx]
test_df = df.iloc[split_idx:].copy()
print(f"train: {len(train_df)} rows ({train_df.shape[0]/len(df):.0%})  |  test: {len(test_df)} rows")

X_train, y_train = train_df[feature_cols], train_df["label"]
X_test, y_test = test_df[feature_cols], test_df["label"]

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, roc_auc_score

model = GradientBoostingClassifier(n_estimators=100, max_depth=3, learning_rate=0.05, random_state=42)
model.fit(X_train, y_train)

pred_proba = model.predict_proba(X_test)[:, 1]
pred = (pred_proba >= 0.5).astype(int)

print(classification_report(y_test, pred))
print("AUC:", roc_auc_score(y_test, pred_proba))

## Does it actually make money? (the question that matters)

Accuracy and AUC measure whether the model separates winners from losers -
they don't say whether trading on that separation is profitable, because
the payout is asymmetric (R:R, not 50/50 coin-flip stakes). A model can
have "61% accuracy" and still lose money if it's not the right 61%.
Compare total R-multiple of taking every raw breakout (the baseline
`orb_datagen.py` was generating) against only taking trades above a
confidence threshold - **on the held-out test set only**, since anything
measured on the training set is not evidence of anything.

In [ ]:
REWARD_RISK = 1.0  # must match orb.py / orb_datagen.py's REWARD_RISK

test_df["r"] = np.where(test_df["label"] == 1, REWARD_RISK, -1.0)
test_df["pred_proba"] = pred_proba

baseline_r = test_df["r"].sum()
baseline_n = len(test_df)
print(f"Baseline - take every raw breakout: {baseline_n} trades, {baseline_r:+.2f}R total, {baseline_r/baseline_n:+.3f}R/trade\n")

for threshold in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75, 0.80]:
    filtered = test_df[test_df["pred_proba"] >= threshold]
    if len(filtered) == 0:
        print(f"threshold {threshold:.2f}: 0 trades (too strict for this test set)")
        continue
    total_r = filtered["r"].sum()
    print(f"threshold {threshold:.2f}: {len(filtered):4d} trades, {total_r:+7.2f}R total, {total_r/len(filtered):+.3f}R/trade")

## Feature importance

Sanity check: do the important features make sense, or is the model
latching onto something spurious (e.g. `day_of_week` dominating would be
suspicious - that's a classic overfitting-to-noise red flag on a dataset
this size)?

In [ ]:
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
importances

## Save the model

`orb_ml.py` loads this at `Initialize()` and calls `predict_proba()` on
each raw breakout's features, only taking the trade above
`ML_CONFIDENCE_THRESHOLD` - pick that threshold from the R-multiple sweep
above, not from whichever number looks biggest without checking trade
count (a threshold with 3 test-set trades is not a threshold, it's noise).

In [ ]:
import pickle

model_bytes = pickle.dumps(model)
qb.ObjectStore.SaveBytes("orb_ml_model", model_bytes)
print("Saved trained model to Object Store key 'orb_ml_model'")